# 08 — Full Evaluation & Comparison
Load results from all CL methods and generate the final comparison tables and plots.

**Requires:** Notebooks 02–07 to have been run and saved results to Drive.

**Outputs:**
- Accuracy matrix per method
- Forgetting / BWT / FWT per method
- Teacher vs student comparison
- All plots saved to Drive

## 0 — Configuration

In [ ]:
import os
os.environ["GITHUB_TOKEN"] = "ghp_YOURTOKEN"
os.environ["BRANCH"]       = "kd&mamal"
DRIVE_PROJECT = "/content/drive/MyDrive/apai"
RESULTS_DIR   = f"{DRIVE_PROJECT}/results"
PLOTS_DIR     = f"{DRIVE_PROJECT}/results/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)
print("Config set.")

## 1 — Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import subprocess, sys
!cp "{DRIVE_PROJECT}/bash/fetch_src.sh" /content/fetch_src.sh
subprocess.run(["bash", "/content/fetch_src.sh"], check=True)
sys.path.insert(0, "/content")
print("Ready.")

## 2 — Imports

In [ ]:
import json, numpy as np, torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from src.utils.metrics import (
    build_accuracy_matrix, average_accuracy, forgetting_measure,
    backward_transfer, forward_transfer, cl_report
)

TASK_LABELS = ["Task 0\n(base)", "Task 1", "Task 2", "Task 3", "Task 4"]
METHODS     = ["Naive", "EWC", "Rehearsal+LwF", "Reptile"]
print("Imports OK.")

## 3 — Load Results
> Adjust paths to match where each notebook saved its results.

In [ ]:
# Each method should have saved a dict: {task_idx -> {task_j -> acc}}
# Example structure (fill in from your saved JSON files):

def load_results(path):
    if not os.path.exists(path):
        print(f"  [!] Not found: {path}")
        return None
    with open(path) as f:
        raw = json.load(f)
    # Convert string keys back to int
    return [{int(j): v for j, v in d.items()} for d in raw]

all_results = {}
for method in METHODS:
    path = f"{RESULTS_DIR}/{method.lower().replace('+','_')}/accuracy_matrix.json"
    res  = load_results(path)
    if res:
        all_results[method] = res
        print(f"  [OK] {method}: {len(res)} tasks")
    else:
        print(f"  [--] {method}: no results yet")

## 4 — Accuracy Matrices & Metrics

In [ ]:
summary = {}
for method, results in all_results.items():
    R = build_accuracy_matrix(results)
    metrics = cl_report(R, method_name=method, class_names_per_task=TASK_LABELS)
    summary[method] = metrics
    print()

## 5 — Comparison Table

In [ ]:
if summary:
    W = 72
    print("=" * W)
    print(f"  {'Method':<20} | {'AA':>8} | {'Forgetting':>11} | {'BWT':>8} | {'FWT':>8}")
    print("-" * W)
    for method, m in summary.items():
        print(f"  {method:<20} | {m['AA']:>8.2%} | {m['F']:>11.2%} | "
              f"{m['BWT']:>+8.2%} | {m['FWT']:>+8.2%}")
    print("=" * W)
else:
    print("No results loaded yet — run notebooks 03–07 first.")

## 6 — Plots

In [ ]:
if summary:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    methods = list(summary.keys())
    x = range(len(methods))

    axes[0].bar(x, [summary[m]["AA"]*100  for m in methods], color="steelblue", alpha=0.8)
    axes[0].set_title("Average Accuracy"); axes[0].set_ylabel("%")
    axes[0].set_xticks(x); axes[0].set_xticklabels(methods, rotation=15)

    axes[1].bar(x, [summary[m]["F"]*100   for m in methods], color="tomato", alpha=0.8)
    axes[1].set_title("Forgetting Measure"); axes[1].set_ylabel("%")
    axes[1].set_xticks(x); axes[1].set_xticklabels(methods, rotation=15)

    axes[2].bar(x, [summary[m]["BWT"]*100 for m in methods],
                color=["green" if summary[m]["BWT"]>=0 else "red" for m in methods], alpha=0.8)
    axes[2].axhline(0, color="k", lw=0.8, ls="--")
    axes[2].set_title("Backward Transfer (BWT)"); axes[2].set_ylabel("pp")
    axes[2].set_xticks(x); axes[2].set_xticklabels(methods, rotation=15)

    plt.suptitle("CL Method Comparison — UCF101 (50 classes, 5 tasks)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/cl_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOTS_DIR}/cl_comparison.png")
else:
    print("No data to plot yet.")

## 7 — Save Summary to Drive

In [ ]:
import json as _json

summary_path = f"{RESULTS_DIR}/final_summary.json"
with open(summary_path, "w") as f:
    _json.dump({m: {k: float(v) for k,v in s.items()} for m,s in summary.items()}, f, indent=2)
print(f"Saved: {summary_path}")